# Dataset

The dataset used is a compilation of 2126 BBC news articles ranging in different topics (politics, business, entertainment, sport and tech) from here: https://www.kaggle.com/datasets/alfathterry/bbc-full-text-document-classification, retrieved on November 1, 2025.

## Tokenization and N-gram generation

In [3]:
import pandas as pd

# Function to load and preprocess the BBC News Summary dataset
def preprocessing_corpus(sourcepath, word_or_character="word", N=2):

	df = pd.read_csv(sourcepath)
	df.columns = ["Article", "Category"]

	# Convert all text to lowercase
	#df['Article'] = df['Article'].str.lower()

	# Replace unicode literal for pound sign
	df = df.replace("xc2xa3", "£", regex=True)

	if word_or_character == "word":
		# Add String "<s> " at the beginning of each text
		df['Article'] = "<s> " + df['Article']

		# For all texts start cleaning by replacing single, double or triple full stops characters with "</s> <s>"
		df = df.replace(r"\.{1,3}", " </s> <s> ", regex=True)
		df = df.replace(r"\!{1,3}", " </s> <s> ", regex=True)
		df = df.replace(r"\?{1,3}", " </s> <s> ", regex=True)
		df = df.replace(r"\ {2,3}", " </s> <s> ", regex=True)

		# Remove all appearances of empty sentences "<s> </s>" and remove the last sentence start token "<s>" at the end of each text
		df = df.replace("<s> </s>", "", regex=True)
		df['Article'] = df['Article'].str[:-6]

		# Remove all appearances of the characters ( ) [ ] { } , " : ; from the texts
		chars_to_remove = [r"\(", r"\)", r"\[", r"\]", r"\{", r"\}", r"\,", r'"', r"\:", r"\;"]
		for char in chars_to_remove:
			df = df.replace(char, "", regex=True)

		# Replace multiple spaces (1, 2 or 3) with a single space	
		df = df.replace(r"\ {1,3}", " ", regex=True)

		# Split each text into a list of words
		words = lambda text: text.split(" ")
		df['Article'] = df['Article'].apply(words)

	else:
		# Replace multiple spaces (1, 2 or 3) with a single space	
		df = df.replace(r"\ {2,3}", " ", regex=True)
		
		# Split each text into a list of characters
		chars = lambda text: list(text)
		df['Article'] = df['Article'].apply(chars)

	# Generate n-grams and store them in a new DataFrame		
	ngram_list = []
	category_list = []

	for index, row in df.iterrows():
		category = row['Category']
		article = row['Article']
		
		# Generate all n-grams for this article at once
		for i in range(len(article) - N + 1):
			ngram = article[i:i+N]
			ngram_list.append(ngram)
			category_list.append(category)

		print(f"Processing corpus for {N}-gram: {index/(len(df)-1)*100:.2f}% processed", end="\r")
	
	ngrams = pd.DataFrame({'N-gram': ngram_list, 'Category': category_list})
    
	return ngrams

In [5]:
# Preprocess text corpus and store ngrams to a new CSV file
N = {1, 2, 3, 4, 5, 6, 7, 8, 9, 10}
modeltype = {"word", "character"}
sourcepath = "./../Dataset/bbc_data.csv"
targetfolder = "./../Dataset_NGrams/"
for n in N:
    for mt in modeltype:
        ngrams = preprocessing_corpus(sourcepath, mt, n)
        ngrams.to_csv(f"{targetfolder}bbc_{mt}_{n}grams.csv", index=False)

## Frequencies

- Frequency dictionary from the N-gram CSV files generated earlier
- The goal is to count how often each possible continuation occurs after a given prefix.


In [8]:
from collections import defaultdict
from ast import literal_eval  # to convert string list ↦ python list

# Function to create the inner dictionary (counts)
def create_inner_dict():

    # It returns a defaultdict(int), where missing keys default to 0.
    return defaultdict(int)


def build_ngram_frequency_dict(csv_path):
    """
    Build a nested frequency dictionary from an N-gram CSV file

    Returns a structure like:
        {
          ('I','am'): {'tired': 5, 'very': 3, '<TOTAL>': 8},
          ('I','will'): {'go': 10, '<TOTAL>': 10}
        }
    """

    # The outer dictionary:
    # - Missing prefix keys will call create_inner_dict()
    # - Which gives us a defaultdict(int)
    freq_dict = defaultdict(create_inner_dict)

    # Load CSV
    df = pd.read_csv(csv_path)

    # Process each N-gram row
    for ngram_str in df["N-gram"]:

        # Convert "['I','am','very']" → ['I','am','very']
        ngram = literal_eval(ngram_str)

        prefix = tuple(ngram[:-1])   # all except last element
        next_token = ngram[-1]       # last element

        # Count the next token
        freq_dict[prefix][next_token] += 1

        # Count total
        freq_dict[prefix]["<TOTAL>"] += 1

    return freq_dict

### Unigram vs. N-gram 

#### Unigram
- A unigram has no prefix — it consists of a single token
- For unigrams, the frequency dictionary counts how many times each token appears

#### N-gram

For example, in a trigram (3-gram) model:

- Prefix = (“I”, “am”)
- Next token = “tired”
- If the sequence "I am tired" appears 5 times in the corpus, then: ("I", "am") → { "tired": 5 }

For all n-grams, the first N − 1 elements form a prefix

The frequency dictionary stores, for each prefix:

- all possible next tokens
- how many times each continuation occurs
- the total count of sequences beginning with that prefix

This allows us to later build full probability models for text generation or language modeling:

$$
P(\text{next token} \mid \text{prefix})
\;=\;
\frac{\text{count(prefix → next)}}{\text{TOTAL(prefix)}}
$$


In [13]:
csv_file = "./../Dataset_NGrams/bbc_word_1grams.csv"

freq = build_ngram_frequency_dict(csv_file)

In [ ]:
list(freq.items())[:5]
#freq[("I", "am")]

[((),
  defaultdict(int,
              {'<s>': 47287,
               '<TOTAL>': 951967,
               'Musicians': 8,
               'to': 24886,
               'tackle': 84,
               'US': 1568,
               'red': 62,
               'tape': 15,
               '</s>': 47244,
               'groups': 154,
               'are': 4384,
               'visa': 36,
               'regulations': 28,
               'which': 2581,
               'blamed': 46,
               'for': 8701,
               'hindering': 1,
               'British': 532,
               'acts': 46,
               'chances': 77,
               'of': 19865,
               'succeeding': 5,
               'across': 222,
               'the': 44583,
               'Atlantic': 17,
               'A': 1024,
               'singer': 117,
               'hoping': 86,
               'perform': 47,
               'in': 16603,
               'can': 1525,
               'expect': 103,
               'pay': 269,
           